# cli

> Simple CLI for LLM agents — fetch, search, and read commands.

In [ ]:
#| default_exp cli

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
"""CLI for fossick — fetch, search, and read commands for LLM agents.
Default output is readable markdown; pass --as-json for JSON (piping/harnesses)."""

import json, sys
from fastcore.script import call_parse
from fossick.core import (fetch as _fetch, to_md, read_arxiv as _read_arxiv,
                          read_yt as _read_yt, search_yt as _search_yt, mv_skill_md, repo_root)
from fossick.search import search as _search

In [ ]:
#| export
@call_parse
def fetch(
    url:str,              # URL to fetch
    sel:str=None,         # CSS selector to extract (None = full page)
    heavy:bool=False,     # JS rendering via headless browser
    stealthy:bool=False,  # Anti-bot stealth fetcher
    as_json:bool=False,   # Output JSON instead of markdown
):
    "Fetch a URL and print as markdown."
    page = _fetch(url, sel=sel, heavy=heavy, stealthy=stealthy)
    if as_json: print(json.dumps({'url': page['url'], 'status': page['status'], 'content': to_md(page)}))
    else: print(to_md(page))

In [ ]:
#| export
@call_parse
def search(
    query:str,            # search query
    n:int=10,             # number of results
    as_json:bool=False,   # output JSON
):
    "Search the web via SearXNG; prints title, URL, and snippet."
    results = _search(query, n=n)
    if as_json: print(json.dumps([dict(r) for r in results], default=str))
    else:
        for r in results:
            print(f"## {r.title}")
            print(f"URL: {r.url}")
            if r.snippet: print(r.snippet[:200])
            print()

In [ ]:
#| export
@call_parse
def read_arxiv(
    url:str,              # arXiv URL or paper ID
    source:bool=False,    # include full paper text
    chars:int=4000,       # max source chars to include
    as_json:bool=False,   # output JSON
):
    "Fetch arXiv paper metadata, authors, and summary."
    p = _read_arxiv(url)
    if as_json:
        out = {k: v for k, v in p.items() if k != 'source'}
        if source: out['source'] = (p.get('source') or '')[:chars]
        print(json.dumps(out, default=str))
    else:
        print(f"# {p['title']}")
        print(f"Authors: {', '.join(p.get('authors', []))}")
        print(f"Published: {str(p.get('published', ''))[:10]}")
        print(f"\n{p.get('summary', '')}")
        if source and p.get('source'): print(f"\n---\n{p['source'][:chars]}")

In [ ]:
#| export
@call_parse
def read_yt(
    url:str,              # YouTube URL or video ID
    as_json:bool=False,   # output JSON
):
    "Fetch YouTube metadata and full transcript."
    v = _read_yt(url)
    if as_json: print(json.dumps(v, default=str))
    else:
        print(f"# {v['title']}")
        print(f"Channel: {v['channel']}  Duration: {v.get('duration')}s")
        if v.get('source'): print(f"\n{v['source']}")

In [ ]:
#| export
@call_parse
def search_yt(
    query:str,            # search query
    n:int=10,             # number of results
    as_json:bool=False,   # output JSON
):
    "Search YouTube; prints title, URL, and channel."
    results = _search_yt(query, n=n)
    if as_json: print(json.dumps(list(results), default=str))
    else:
        for r in results:
            print(f"## {r['title']}")
            print(f"URL: {r['url']}  Channel: {r.get('channel', '')}  Duration: {r.get('duration')}s")
            if r.get('description'): print(r['description'][:150])
            print()

In [ ]:
#| export
@call_parse
def install():
    "Install fossick SKILL.md to .agents/skills/fossick/ and .claude/skills/fossick/."
    mv_skill_md(dry_run=False, dir=repo_root())

In [ ]:
#| export
@call_parse
def sniff(
    url:str,              # URL to navigate to in Chrome
    pattern:str='*',      # URL pattern to filter captured requests (glob or regex)
    timeout:int=15,       # seconds to wait for network activity
    as_json:bool=False,   # output JSON
):
    "Capture network requests fired by a page using a real Chrome session."
    from fossick.cdp import automation_browser
    with automation_browser() as s:
        caps = s.sniff(url=url, pattern=pattern, timeout=timeout)
    if as_json:
        print(json.dumps([dict(c) for c in caps], default=str))
    else:
        for c in caps:
            print(f"## {c.url}")
            if c.get('request_body'): print(f"Body: {json.dumps(c.request_body, default=str)[:200]}")
            if c.get('response_body'):
                preview = json.dumps(c.response_body, default=str)[:300] if isinstance(c.response_body, (dict,list)) else str(c.response_body)[:300]
                print(f"Response: {preview}")
            print()

In [ ]:
#| export
CMDS = {
    'fetch':      fetch,
    'search':     search,
    'read-arxiv': read_arxiv,
    'read-yt':    read_yt,
    'search-yt':  search_yt,
    'sniff':      sniff,
    'install':    install,
}

def main():
    "Entry point for the `fossick` CLI command."
    if len(sys.argv) < 2 or sys.argv[1] not in CMDS:
        print(f"Usage: fossick [{' | '.join(CMDS)}]")
        sys.exit(0 if len(sys.argv) < 2 else 1)
    cmd = sys.argv.pop(1)
    CMDS[cmd]()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()